In [1]:
pip install scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from openpyxl import load_workbook
import os
from openpyxl.worksheet.copier import WorksheetCopy

# 🔹 사용자 설정 변수
filtered_filename = '충청북도.xlsx'
road_filter = ''             # 예: '대로'
cont_years = ''        # 예: '23, 24' → 연속된 문자열 허용

# 🔹 파일 경로
file_path = os.path.join(r'C:\startcoding\practice\정제파일\필터링결과', filtered_filename)

# 🔹 Sheet1에서 데이터 읽기
df = pd.read_excel(file_path, sheet_name='Sheet1')

# 🔹 기본 전처리
df = df[['용도지역', '지목', '단가', '도로조건', '연도']].dropna()

# 🔹 조건 필터링
if road_filter:
    df = df[df['도로조건'].str.contains(road_filter)]

if cont_years:
    year_list = [y.strip() for y in cont_years.split(',')]
    df = df[df['연도'].astype(str).str[-2:].isin(year_list)]

# 🔹 통계 함수
def stats(group):
    count = group.count()
    mean = group.mean()
    std = group.std()
    ci_low, ci_high = norm.interval(0.90, loc=mean, scale=std / np.sqrt(count)) if count > 1 else (np.nan, np.nan)
    q1 = group.quantile(0.25)
    median = group.median()
    q3 = group.quantile(0.75)
    min_val = group.min()
    max_val = group.max()
    return pd.Series({
        'count': count, 'mean': mean, 'std': std,
        'ci_high': ci_high, 'ci_low': ci_low,
        'min': min_val, '25%': q1, 'median': median,
        '75%': q3, 'max': max_val
    })

# 🔹 그룹화 및 통계 계산
grouped = df.groupby(['용도지역', '지목'])['단가'].apply(stats).unstack()

# 🔹 정렬 기준
zones = df.groupby('용도지역')['단가'].count().sort_values(ascending=False).index.tolist()
uses = df.groupby('지목')['단가'].count().sort_values(ascending=False).index.tolist()

# 🔹 엑셀 파일 열기
wb = load_workbook(file_path)

# 🔹 템플릿 시트 확보 (분석표 시트가 없으면 새로 생성)
if '분석표' not in wb.sheetnames:
    template_ws = wb.create_sheet('분석표')
else:
    template_ws = wb['분석표']

# 🔹 분석 결과 시트 이름 결정
new_sheet_name = '분석표'
suffix = []

if cont_years:
    suffix.append(f"({cont_years.replace(',', '_')})")
if road_filter:
    suffix.append(f"({road_filter})")

if suffix:
    new_sheet_name = f"분석표{''.join(suffix)}"
    # 기존에 같은 이름이 있다면 삭제 후 생성
    if new_sheet_name in wb.sheetnames:
        del wb[new_sheet_name]
    copied_ws = wb.copy_worksheet(template_ws)
    copied_ws.title = new_sheet_name
else:
    copied_ws = template_ws
    # 값만 초기화
    for row in copied_ws.iter_rows(min_row=2, max_row=copied_ws.max_row,
                                   min_col=1, max_col=copied_ws.max_column):
        for cell in row:
            cell.value = None

# 🔹 헤더 작성 (지목)
for j, use in enumerate(uses):
    copied_ws.cell(row=1, column=2 + j * 2, value=use)

# 🔹 본문 작성
current_row = 2
for zone in zones:
    zone_written = False
    for j, use in enumerate(uses):
        if (zone, use) in grouped.index:
            row_data = grouped.loc[(zone, use)].round(2)
            left = ['count', 'mean', 'std', 'ci_high', 'ci_low']
            right = ['min', '25%', 'median', '75%', 'max']

            for k in range(5):
                if not zone_written and k == 0:
                    copied_ws.cell(row=current_row + k, column=1, value=zone)

                copied_ws.cell(row=current_row + k, column=2 + j * 2, value=row_data[left[k]])
                copied_ws.cell(row=current_row + k, column=2 + j * 2 + 1, value=row_data[right[k]])
            zone_written = True
    current_row += 5

# 🔹 저장
wb.save(file_path)

c:\Users\PC\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\stats\_distn_infrastructure.py:2304: RuntimeWarning: invalid value encountered in multiply
  lower_bound = _a * scale + loc
c:\Users\PC\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\stats\_distn_infrastructure.py:2305: RuntimeWarning: invalid value encountered in multiply
  upper_bound = _b * scale + loc
c:\Users\PC\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\stats\_distn_infrastructure.py:2304: RuntimeWarning: invalid value encountered in multiply
  lower_bound = _a * scale + loc
c:\Users\PC\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\stats\_distn_infrastructure.py:2305: RuntimeWarning: invalid value encountered in multiply
  upper_bound = _b * scale + loc
c:\Users\PC\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\stats\_distn_infrastructure.py:2304: RuntimeWarning: invalid value encountered in multiply
  lower_bound = _a * scale + loc
